In [1]:
# CELL 1 - Environment verification & installation
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
# Colab usually ships WITHOUT OpenMPI: install it first so that `mpirun` and mpi4py work
!apt-get install -y -qq openmpi-bin libopenmpi-dev > /dev/null
!mpirun --allow-run-as-root --version | head -n 2
!pip install -q mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-fd8bb1a0-f2bf-2b19-956a-0e2661ff8a08, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [2]:
# CELL 2 - Student identity configuration
GITHUB_USERNAME = "brainmegamind303-droid"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0, (
    "You must set GITHUB_USERNAME to your real GitHub handle."
)
print(f"Configured student GitHub identity: {GITHUB_USERNAME}")

Configured student GitHub identity: brainmegamind303-droid


In [3]:
%%writefile distributed_cuda_solver.py
import os
import sys
import json
import math
import time
import re
import hashlib
import subprocess
import numpy as np
from mpi4py import MPI
from numba import cuda, float64

# Compile-time constants for 2D shared-memory tiling (16x16 interior + 1-cell halo apron)
TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2  # 18
GLOBAL_NX = 512
GLOBAL_NY = 512
STEPS = 400
LAMBDA_DAMP = 0.15

# Derived constants
PROC_DIMS = (2, 2)
NY_LOC = GLOBAL_NY // PROC_DIMS[0]   # 256 owned rows
NX_LOC = GLOBAL_NX // PROC_DIMS[1]   # 256 owned cols
PAD_NY = NY_LOC + 2                  # 258
PAD_NX = NX_LOC + 2                  # 258
DX = 1.0 / (GLOBAL_NX + 1)           # 1/513
DY = 1.0 / (GLOBAL_NY + 1)           # 1/513
KAPPA_MIN, KAPPA_MAX_BOUND = 0.08, 0.24
CFL_SAFETY = 0.8
DT_MAX_BOUND = DX * DX / (4.0 * KAPPA_MAX_BOUND)
DT = CFL_SAFETY * DT_MAX_BOUND
DIAG_EVERY = 100
WARMUP_STEPS = 4

# Message tags (named by direction of travel of the data)
TAG_UP, TAG_DOWN, TAG_LEFT, TAG_RIGHT = 10, 11, 12, 13


def get_gpu_uuid() -> str:
    """Query active NVIDIA GPU UUID via nvidia-smi."""
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return out.strip().splitlines()[0].strip()


def derive_student_seed(github_user: str, gpu_uuid: str) -> int:
    """Derive deterministic 64-bit integer seed bound to student GitHub handle and GPU UUID."""
    digest = hashlib.sha256(f"{github_user.strip()}::{gpu_uuid}".encode("utf8")).hexdigest()
    return int(digest[:16], 16)


# ==============================================================================
# TASK 1: Distributed Cartesian Topology & Hardware Context Initialization
# ==============================================================================
def init_distributed_context(github_user: str):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Expected exactly 4 MPI ranks (2x2 grid), got {size}.")

    # Bind process to GPU 0 on Google Colab
    cuda.select_device(0)
    dev = cuda.get_current_device()
    dev_name = dev.name.decode() if isinstance(dev.name, bytes) else str(dev.name)
    cc = tuple(dev.compute_capability)

    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = comm.bcast(gpu_uuid, root=0)
    instance_seed = derive_student_seed(github_user, gpu_uuid)

    # Non-periodic 2x2 Cartesian communicator
    cart_comm = comm.Create_cart(dims=list(PROC_DIMS), periods=[False, False], reorder=False)
    cart_rank = cart_comm.Get_rank()
    row_rank, col_rank = cart_comm.Get_coords(cart_rank)

    # Shift(direction, disp=1) -> (source, dest): source has lower coordinate along that axis.
    nbr_up, nbr_down = cart_comm.Shift(0, 1)
    nbr_left, nbr_right = cart_comm.Shift(1, 1)

    return {
        "comm": comm,
        "cart_comm": cart_comm,
        "rank": cart_rank,
        "size": size,
        "row_rank": row_rank,
        "col_rank": col_rank,
        "nbr_up": nbr_up,
        "nbr_down": nbr_down,
        "nbr_left": nbr_left,
        "nbr_right": nbr_right,
        "github_user": github_user.strip(),
        "gpu_uuid": gpu_uuid,
        "instance_seed": instance_seed,
        "device_name": dev_name,
        "compute_capability": f"{cc[0]}.{cc[1]}",
    }


# ==============================================================================
# TASK 2: Seeded Heterogeneous Subdomain Initialization (rank-local arrays only)
# ==============================================================================
def initialize_subdomains(ctx):
    seed = ctx["instance_seed"]
    rng = np.random.default_rng(seed)  # identical on all ranks -> identical global parameters

    # Seed-derived physical parameters (same on every rank, no communication needed)
    pulses = []
    for _ in range(3):
        cx, cy = rng.uniform(0.2, 0.8, size=2)
        amp = rng.uniform(0.8, 1.2)
        sig = rng.uniform(0.05, 0.09)
        pulses.append((cx, cy, amp, sig))
    phase_x, phase_y = rng.uniform(0.0, 2.0 * np.pi, size=2)
    src_cx, src_cy = rng.uniform(0.2, 0.8, size=2)
    src_amp = 0.01 * rng.uniform(0.5, 1.5)
    src_sig = 0.04

    # Global physical coordinates of local indices 0..257 (node g = rank_offset + local index,
    # g = 0 and g = 513 are the physical Dirichlet boundary nodes). No global array allocated.
    gi = ctx["row_rank"] * NY_LOC + np.arange(PAD_NY, dtype=np.float64)
    gj = ctx["col_rank"] * NX_LOC + np.arange(PAD_NX, dtype=np.float64)
    y = gi * DY
    x = gj * DX
    Y, X = np.meshgrid(y, x, indexing="ij")  # shape (258, 258): axis 0 = y (rows), axis 1 = x (cols)

    # Heterogeneous diffusivity in [0.08, 0.24]
    kappa = 0.16 + 0.08 * np.sin(2.0 * np.pi * 2.0 * X + phase_x) * np.cos(2.0 * np.pi * 3.0 * Y + phase_y)
    kappa = np.ascontiguousarray(np.clip(kappa, KAPPA_MIN, KAPPA_MAX_BOUND), dtype=np.float64)

    # Initial Gaussian thermal pulses
    u0 = np.zeros((PAD_NY, PAD_NX), dtype=np.float64)
    for cx, cy, amp, sig in pulses:
        u0 += amp * np.exp(-((X - cx) ** 2 + (Y - cy) ** 2) / (2.0 * sig * sig))

    # Localized source field
    S = src_amp * np.exp(-((X - src_cx) ** 2 + (Y - src_cy) ** 2) / (2.0 * src_sig * src_sig))
    S = np.ascontiguousarray(S, dtype=np.float64)

    # Homogeneous Dirichlet u = 0 on outer global edges (neighbor == PROC_NULL)
    if ctx["nbr_up"] == MPI.PROC_NULL:
        u0[0, :] = 0.0
    if ctx["nbr_down"] == MPI.PROC_NULL:
        u0[PAD_NY - 1, :] = 0.0
    if ctx["nbr_left"] == MPI.PROC_NULL:
        u0[:, 0] = 0.0
    if ctx["nbr_right"] == MPI.PROC_NULL:
        u0[:, PAD_NX - 1] = 0.0
    u0 = np.ascontiguousarray(u0)

    # Global maximum of kappa over owned cells (for the CFL analysis, Q6)
    local_kmax = np.array([kappa[1:NY_LOC + 1, 1:NX_LOC + 1].max()], dtype=np.float64)
    global_kmax = np.zeros(1, dtype=np.float64)
    ctx["cart_comm"].Allreduce(local_kmax, global_kmax, op=MPI.MAX)

    return {
        "u0_host": u0,
        "kappa_host": kappa,
        "S_host": S,
        "d_u_a": cuda.to_device(u0),
        "d_u_b": cuda.to_device(u0),
        "d_kappa": cuda.to_device(kappa),
        "d_S": cuda.to_device(S),
        "kappa_max": float(global_kmax[0]),
        "pulses": [tuple(float(v) for v in p) for p in pulses],
        "source": (float(src_cx), float(src_cy), float(src_amp), float(src_sig)),
    }


# ==============================================================================
# TASK 3: GPU Halo Pack/Unpack Kernels & Pinned MPI Exchange
# Buffer layout (float64, 1D, contiguous):
#   [ up : nx | down : nx | left : ny | right : ny ]   total = 2*nx + 2*ny
# send: up = row 1, down = row ny, left = column 1, right = column nx
# recv: up -> ghost row 0, down -> ghost row ny+1, left -> ghost col 0, right -> ghost col nx+1
# ==============================================================================
@cuda.jit
def pack_halos_kernel(u, sendbuf, ny, nx):
    t = cuda.grid(1)
    total = 2 * nx + 2 * ny
    if t >= total:
        return
    if t < nx:                       # top owned row (row 1)
        sendbuf[t] = u[1, 1 + t]
    elif t < 2 * nx:                 # bottom owned row (row ny)
        k = t - nx
        sendbuf[t] = u[ny, 1 + k]
    elif t < 2 * nx + ny:            # left owned column (column 1)  - strided gather
        k = t - 2 * nx
        sendbuf[t] = u[1 + k, 1]
    else:                            # right owned column (column nx) - strided gather
        k = t - 2 * nx - ny
        sendbuf[t] = u[1 + k, nx]


@cuda.jit
def unpack_halos_kernel(u, recvbuf, ny, nx, has_up, has_down, has_left, has_right):
    t = cuda.grid(1)
    total = 2 * nx + 2 * ny
    if t >= total:
        return
    if t < nx:
        if has_up:
            u[0, 1 + t] = recvbuf[t]
    elif t < 2 * nx:
        if has_down:
            k = t - nx
            u[ny + 1, 1 + k] = recvbuf[t]
    elif t < 2 * nx + ny:
        if has_left:
            k = t - 2 * nx
            u[1 + k, 0] = recvbuf[t]
    else:
        if has_right:
            k = t - 2 * nx - ny
            u[1 + k, nx + 1] = recvbuf[t]


def create_halo_buffers(ny=NY_LOC, nx=NX_LOC):
    total = 2 * nx + 2 * ny
    return {
        "ny": ny,
        "nx": nx,
        "total": total,
        "d_send": cuda.device_array(total, dtype=np.float64),
        "d_recv": cuda.device_array(total, dtype=np.float64),
        "h_send": cuda.pinned_array(total, dtype=np.float64),   # page-locked
        "h_recv": cuda.pinned_array(total, dtype=np.float64),   # page-locked
    }


def exchange_halos_async(ctx, bufs, d_u, stream_boundary):
    """Stage A + start of Stage B: pack on GPU, async D2H into pinned memory,
    synchronize the boundary stream, then post non-blocking Irecv/Isend.
    Returns the list of MPI requests (completed later by finish_halo_exchange)."""
    ny, nx, total = bufs["ny"], bufs["nx"], bufs["total"]
    threads = 128
    blocks = (total + threads - 1) // threads

    # Stage A: pack (strided gather on device) + async D2H DMA on stream_boundary
    pack_halos_kernel[blocks, threads, stream_boundary](d_u, bufs["d_send"], ny, nx)
    bufs["d_send"].copy_to_host(bufs["h_send"], stream=stream_boundary)
    stream_boundary.synchronize()  # host send buffers are now valid

    h_send, h_recv = bufs["h_send"], bufs["h_recv"]
    off_up, off_down, off_left, off_right = 0, nx, 2 * nx, 2 * nx + ny

    cart = ctx["cart_comm"]
    reqs = []
    # Receives (data travelling toward me)
    reqs.append(cart.Irecv(h_recv[off_up:off_up + nx], source=ctx["nbr_up"], tag=TAG_DOWN))
    reqs.append(cart.Irecv(h_recv[off_down:off_down + nx], source=ctx["nbr_down"], tag=TAG_UP))
    reqs.append(cart.Irecv(h_recv[off_left:off_left + ny], source=ctx["nbr_left"], tag=TAG_RIGHT))
    reqs.append(cart.Irecv(h_recv[off_right:off_right + ny], source=ctx["nbr_right"], tag=TAG_LEFT))
    # Sends
    reqs.append(cart.Isend(h_send[off_up:off_up + nx], dest=ctx["nbr_up"], tag=TAG_UP))
    reqs.append(cart.Isend(h_send[off_down:off_down + nx], dest=ctx["nbr_down"], tag=TAG_DOWN))
    reqs.append(cart.Isend(h_send[off_left:off_left + ny], dest=ctx["nbr_left"], tag=TAG_LEFT))
    reqs.append(cart.Isend(h_send[off_right:off_right + ny], dest=ctx["nbr_right"], tag=TAG_RIGHT))
    return reqs


def finish_halo_exchange(ctx, bufs, d_u, reqs, stream_boundary):
    """Stage C (first half): Waitall, async H2D on stream_boundary, unpack ghost cells."""
    MPI.Request.Waitall(reqs)
    bufs["d_recv"].copy_to_device(bufs["h_recv"], stream=stream_boundary)
    threads = 128
    blocks = (bufs["total"] + threads - 1) // threads
    unpack_halos_kernel[blocks, threads, stream_boundary](
        d_u, bufs["d_recv"], bufs["ny"], bufs["nx"],
        ctx["nbr_up"] != MPI.PROC_NULL,
        ctx["nbr_down"] != MPI.PROC_NULL,
        ctx["nbr_left"] != MPI.PROC_NULL,
        ctx["nbr_right"] != MPI.PROC_NULL,
    )


# ==============================================================================
# TASK 4: 2D Shared-Memory Tiled Stencil & Dual-Stream Orchestrator
# Bounding box is INCLUSIVE: updates cells with r_start<=i<=r_end, c_start<=j<=c_end
# ==============================================================================
@cuda.jit
def tiled_stencil_kernel(u, un, kappa, S, r_start, r_end, c_start, c_end,
                         dt, inv_dx2, inv_dy2, lam):
    smem = cuda.shared.array(shape=(SHARED_DIM, SHARED_DIM), dtype=float64)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    i = r_start + cuda.blockIdx.y * TILE_DIM + ty
    j = c_start + cuda.blockIdx.x * TILE_DIM + tx
    nrows = u.shape[0]
    ncols = u.shape[1]

    # Center cell
    if i < nrows and j < ncols:
        smem[ty + 1, tx + 1] = u[i, j]
    # 1-cell halo apron (edge threads of the block)
    if ty == 0 and i >= 1 and j < ncols:
        smem[0, tx + 1] = u[i - 1, j]
    if ty == TILE_DIM - 1 and i + 1 < nrows and j < ncols:
        smem[SHARED_DIM - 1, tx + 1] = u[i + 1, j]
    if tx == 0 and j >= 1 and i < nrows:
        smem[ty + 1, 0] = u[i, j - 1]
    if tx == TILE_DIM - 1 and j + 1 < ncols and i < nrows:
        smem[ty + 1, SHARED_DIM - 1] = u[i, j + 1]

    cuda.syncthreads()  # every thread reaches the barrier (no early return above)

    if i >= r_start and i <= r_end and j >= c_start and j <= c_end:
        c = smem[ty + 1, tx + 1]
        un_c = smem[ty + 2, tx + 1]   # i+1
        us_c = smem[ty, tx + 1]       # i-1
        ue_c = smem[ty + 1, tx + 2]   # j+1
        uw_c = smem[ty + 1, tx]       # j-1

        k0 = kappa[i, j]
        k_n = 0.5 * (kappa[i + 1, j] + k0)
        k_s = 0.5 * (kappa[i - 1, j] + k0)
        k_e = 0.5 * (kappa[i, j + 1] + k0)
        k_w = 0.5 * (kappa[i, j - 1] + k0)

        div_y = (k_n * (un_c - c) - k_s * (c - us_c)) * inv_dy2
        div_x = (k_e * (ue_c - c) - k_w * (c - uw_c)) * inv_dx2
        un[i, j] = c + dt * (div_x + div_y - lam * c * c * c + S[i, j])


def launch_stencil(stream, fld, d_in, d_out, r0, r1, c0, c1):
    nr = r1 - r0 + 1
    nc = c1 - c0 + 1
    if nr <= 0 or nc <= 0:
        return
    grid = ((nc + TILE_DIM - 1) // TILE_DIM, (nr + TILE_DIM - 1) // TILE_DIM)
    tiled_stencil_kernel[grid, (TILE_DIM, TILE_DIM), stream](
        d_in, d_out, fld["d_kappa"], fld["d_S"],
        r0, r1, c0, c1,
        float(DT), float(1.0 / (DX * DX)), float(1.0 / (DY * DY)), float(LAMBDA_DAMP),
    )


def step_overlapped(ctx, fld, bufs, streams, d_in, d_out):
    """One dual-stream step: interior compute overlaps with MPI halo exchange."""
    ny, nx = NY_LOC, NX_LOC
    stream_interior, stream_boundary = streams

    # Stage A + B(post): pack, async D2H, sync boundary stream, post Irecv/Isend
    reqs = exchange_halos_async(ctx, bufs, d_in, stream_boundary)

    # Stage B (concurrent): deep interior [2..ny-1, 2..nx-1] on stream_interior
    launch_stencil(stream_interior, fld, d_in, d_out, 2, ny - 1, 2, nx - 1)

    # Stage C: wait, async H2D, unpack ghosts, boundary ring stencil on stream_boundary
    finish_halo_exchange(ctx, bufs, d_in, reqs, stream_boundary)
    launch_stencil(stream_boundary, fld, d_in, d_out, 1, 1, 1, nx)             # row 1
    launch_stencil(stream_boundary, fld, d_in, d_out, ny, ny, 1, nx)           # row ny
    launch_stencil(stream_boundary, fld, d_in, d_out, 2, ny - 1, 1, 1)         # column 1
    launch_stencil(stream_boundary, fld, d_in, d_out, 2, ny - 1, nx, nx)       # column nx

    stream_boundary.synchronize()
    stream_interior.synchronize()


def step_serialized(ctx, fld, bufs, streams, d_in, d_out):
    """Reference step: fully serialized exchange -> full-interior stencil on a single stream."""
    ny, nx = NY_LOC, NX_LOC
    _, stream_boundary = streams

    reqs = exchange_halos_async(ctx, bufs, d_in, stream_boundary)
    MPI.Request.Waitall(reqs)  # explicit serialization: nothing computes while messages are in flight
    finish_halo_exchange(ctx, bufs, d_in, [], stream_boundary)
    launch_stencil(stream_boundary, fld, d_in, d_out, 1, ny, 1, nx)  # entire owned interior
    stream_boundary.synchronize()


# ==============================================================================
# TASK 5: Distributed Reductions, PTX Audit & Comparative Benchmark
# ==============================================================================
def global_diagnostics(ctx, d_new, d_old, step):
    """Global thermal energy E = sum u*dx*dy, L2 norm ||u||_2 and L2 residual ||(u^n - u^{n-1})/dt||_2.
    Computed with Allreduce(SUM) over a 3-vector (no field gather)."""
    new_h = d_new.copy_to_host()[1:NY_LOC + 1, 1:NX_LOC + 1]
    local = np.zeros(3, dtype=np.float64)
    local[0] = new_h.sum() * DX * DY
    local[1] = (new_h * new_h).sum() * DX * DY
    if step > 0:
        old_h = d_old.copy_to_host()[1:NY_LOC + 1, 1:NX_LOC + 1]
        diff = (new_h - old_h) / DT
        local[2] = (diff * diff).sum() * DX * DY
    glob = np.zeros(3, dtype=np.float64)
    ctx["cart_comm"].Allreduce(local, glob, op=MPI.SUM)
    return {
        "step": step,
        "energy": float(glob[0]),
        "l2_norm": float(math.sqrt(glob[1])),
        "l2_residual": float(math.sqrt(glob[2])),
    }


def reset_state(fld):
    fld["d_u_a"].copy_to_device(fld["u0_host"])
    fld["d_u_b"].copy_to_device(fld["u0_host"])


def run_mode(ctx, fld, bufs, streams, mode, steps=STEPS):
    stepper = step_overlapped if mode == "overlapped" else step_serialized
    cart = ctx["cart_comm"]

    # Warm-up (JIT compilation + MPI/CUDA warm paths), then restore identical initial state
    reset_state(fld)
    u_cur, u_nxt = fld["d_u_a"], fld["d_u_b"]
    for _ in range(WARMUP_STEPS):
        stepper(ctx, fld, bufs, streams, u_cur, u_nxt)
        u_cur, u_nxt = u_nxt, u_cur
    cuda.synchronize()
    reset_state(fld)
    u_cur, u_nxt = fld["d_u_a"], fld["d_u_b"]

    trace = [global_diagnostics(ctx, u_cur, None, 0)]
    wall_ms = 0.0
    event_ms = 0.0
    cart.Barrier()

    done = 0
    while done < steps:
        chunk = min(DIAG_EVERY, steps - done)
        ev_start, ev_end = cuda.event(), cuda.event()
        cart.Barrier()
        t0 = time.perf_counter()
        ev_start.record(stream=streams[1])
        for _ in range(chunk):
            stepper(ctx, fld, bufs, streams, u_cur, u_nxt)
            u_cur, u_nxt = u_nxt, u_cur
        ev_end.record(stream=streams[1])
        ev_end.synchronize()
        t1 = time.perf_counter()
        wall_ms += (t1 - t0) * 1e3
        event_ms += ev_start.elapsed_time(ev_end)
        done += chunk
        # after swap: u_cur = u^{done}, u_nxt = u^{done-1}
        trace.append(global_diagnostics(ctx, u_cur, u_nxt, done))

    final_interior = u_cur.copy_to_host()[1:NY_LOC + 1, 1:NX_LOC + 1].copy()
    return {"trace": trace, "wall_ms": wall_ms, "event_ms": event_ms, "final": final_interior}


def ptx_audit():
    """Inspect PTX emitted for tiled_stencil_kernel (must be called after it has been launched)."""
    asm = tiled_stencil_kernel.inspect_asm()
    if isinstance(asm, dict):
        ptx = "\n".join(asm.values())
    else:
        ptx = str(asm)
    counts = {
        "st.shared.f64": len(re.findall(r"st\.shared(?:\.\w+)*\.f64", ptx)),
        "ld.shared.f64": len(re.findall(r"ld\.shared(?:\.\w+)*\.f64", ptx)),
        "bar.sync": len(re.findall(r"bar\.sync", ptx)),
        "st.shared (any)": len(re.findall(r"st\.shared", ptx)),
        "ld.shared (any)": len(re.findall(r"ld\.shared", ptx)),
    }
    passed = counts["st.shared (any)"] > 0 and counts["ld.shared (any)"] > 0 and counts["bar.sync"] > 0
    return counts, passed


def run_solver(ctx):
    cart = ctx["cart_comm"]
    rank = ctx["rank"]

    fld = initialize_subdomains(ctx)
    bufs = create_halo_buffers()
    streams = (cuda.stream(), cuda.stream())  # (stream_interior, stream_boundary)

    kmax = fld["kappa_max"]
    dt_max = DX * DX / (4.0 * kmax)
    if rank == 0:
        print("=" * 78)
        print(f"GITHUB_USERNAME     : {ctx['github_user']}")
        print(f"GPU                 : {ctx['device_name']} (CC {ctx['compute_capability']})")
        print(f"GPU UUID            : {ctx['gpu_uuid']}")
        print(f"INSTANCE_SEED       : {ctx['instance_seed']}")
        print(f"MPI ranks           : {ctx['size']} in a {PROC_DIMS[0]}x{PROC_DIMS[1]} non-periodic Cartesian grid")
        print(f"Local padded array  : ({PAD_NY}, {PAD_NX}) float64 | owned {NY_LOC}x{NX_LOC}")
        print(f"dx = dy = 1/513     : {DX:.9e}")
        print(f"kappa_max (run)     : {kmax:.9f}")
        print(f"dt_max = dx^2/(4k)  : {dt_max:.6e}")
        print(f"dt used             : {DT:.6e}  (dt/dt_max = {DT / dt_max:.4f})")
        print("=" * 78)
    for r in range(ctx["size"]):
        cart.Barrier()
        if r == rank:
            print(f"[rank {rank}] coords=({ctx['row_rank']},{ctx['col_rank']}) up={ctx['nbr_up']} "
                  f"down={ctx['nbr_down']} left={ctx['nbr_left']} right={ctx['nbr_right']}", flush=True)
        cart.Barrier()

    res_ser = run_mode(ctx, fld, bufs, streams, "serialized")
    res_ovl = run_mode(ctx, fld, bufs, streams, "overlapped")

    # Parity between serialized and overlapped final states (global max-abs difference)
    local_diff = np.array([np.max(np.abs(res_ser["final"] - res_ovl["final"]))], dtype=np.float64)
    parity = np.zeros(1, dtype=np.float64)
    cart.Allreduce(local_diff, parity, op=MPI.MAX)
    parity_max_diff = float(parity[0])

    # Worst-case (slowest rank) timings
    t_loc = np.array([res_ser["wall_ms"], res_ovl["wall_ms"], res_ser["event_ms"], res_ovl["event_ms"]])
    t_max = np.zeros(4)
    cart.Allreduce(t_loc, t_max, op=MPI.MAX)

    counts, ptx_ok = ptx_audit()

    if rank == 0:
        for name, res in (("serialized", res_ser), ("overlapped", res_ovl)):
            print(f"\nALLREDUCE TRACE [{name}]")
            for t in res["trace"]:
                print(f"  step {t['step']:4d} | E = {t['energy']:.10e} | "
                      f"||u||_2 = {t['l2_norm']:.10e} | L2 residual = {t['l2_residual']:.10e}")
        energies = [t["energy"] for t in res_ovl["trace"]]
        monotonic = all(energies[k + 1] < energies[k] for k in range(len(energies) - 1))
        print(f"\nEnergy monotonically decreasing (overlapped): {monotonic}")
        print(f"\nserialized_ms  (wall, slowest rank) : {t_max[0]:.3f}")
        print(f"overlapped_ms  (wall, slowest rank) : {t_max[1]:.3f}")
        print(f"serialized CUDA-event ms            : {t_max[2]:.3f}")
        print(f"overlapped CUDA-event ms            : {t_max[3]:.3f}")
        print(f"overlap speedup (wall)              : {t_max[0] / t_max[1]:.4f}x")
        print(f"\nparity_max_diff (serialized vs overlapped) : {parity_max_diff:.3e}  "
              f"[{'PASS' if parity_max_diff < 1e-12 else 'FAIL'}]")
        print("\nPTX AUDIT (tiled_stencil_kernel.inspect_asm())")
        for k, v in counts.items():
            print(f"  {k:18s}: {v}")
        print(f"  ptx_audit_passed  : {ptx_ok}")

        manifest = {
            "github_username": ctx["github_user"],
            "gpu_uuid": ctx["gpu_uuid"],
            "gpu_name": ctx["device_name"],
            "compute_capability": ctx["compute_capability"],
            "instance_seed": ctx["instance_seed"],
            "mpi_ranks": ctx["size"],
            "cartesian_dims": list(PROC_DIMS),
            "global_grid": [GLOBAL_NY, GLOBAL_NX],
            "local_padded_shape": [PAD_NY, PAD_NX],
            "steps": STEPS,
            "dx": DX,
            "dt": DT,
            "kappa_max": kmax,
            "dt_max": dt_max,
            "lambda_damp": LAMBDA_DAMP,
            "serialized_ms": t_max[0],
            "overlapped_ms": t_max[1],
            "serialized_cuda_event_ms": t_max[2],
            "overlapped_cuda_event_ms": t_max[3],
            "parity_max_diff": parity_max_diff,
            "parity_passed": parity_max_diff < 1e-12,
            "ptx_counts": counts,
            "ptx_audit_passed": bool(ptx_ok),
            "energy_trace_serialized": res_ser["trace"],
            "energy_trace_overlapped": res_ovl["trace"],
            "energy_monotonic_decreasing": bool(monotonic),
        }
        with open("submission_manifest.json", "w") as f:
            json.dump(manifest, f, indent=2)
        print("\nsubmission_manifest.json written.")
        print("ALL TASKS COMPLETED.")

    return parity_max_diff, counts, ptx_ok


def main():
    if len(sys.argv) < 2:
        raise SystemExit("Usage: mpirun -np 4 python3 distributed_cuda_solver.py <github_username>")
    ctx = init_distributed_context(sys.argv[1])
    run_solver(ctx)


if __name__ == "__main__":
    main()


Writing distributed_cuda_solver.py


In [4]:
# CELL 4 - Multi-rank MPI + CUDA execution launcher
import subprocess
cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py",
    GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.stderr.strip():
    print("STDERR:\n", proc.stderr)
if proc.returncode != 0:
    raise RuntimeError(f"MPI execution failed with exit code {proc.returncode}")

GITHUB_USERNAME     : brainmegamind303-droid
GPU                 : Tesla T4 (CC 7.5)
GPU UUID            : GPU-fd8bb1a0-f2bf-2b19-956a-0e2661ff8a08
INSTANCE_SEED       : 2404270034748271502
MPI ranks           : 4 in a 2x2 non-periodic Cartesian grid
Local padded array  : (258, 258) float64 | owned 256x256
dx = dy = 1/513     : 1.949317739e-03
kappa_max (run)     : 0.239997279
dt_max = dx^2/(4k)  : 3.958211e-06
dt used             : 3.166533e-06  (dt/dt_max = 0.8000)
[rank 0] coords=(0,0) up=-2 down=2 left=-2 right=1
[rank 1] coords=(0,1) up=-2 down=3 left=0 right=-2
[rank 2] coords=(1,0) up=0 down=-2 left=-2 right=3
[rank 3] coords=(1,1) up=1 down=-2 left=2 right=-2

ALLREDUCE TRACE [serialized]
  step    0 | E = 8.4028474940e-02 | ||u||_2 = 2.4324633554e-01 | L2 residual = 0.0000000000e+00
  step  100 | E = 8.4018321079e-02 | ||u||_2 = 2.4096231242e-01 | L2 residual = 1.0535289717e+01
  step  200 | E = 8.4010680184e-02 | ||u||_2 = 2.3875010444e-01 | L2 residual = 1.0177060016e+01
  s

In [5]:
# CELL 4b - Show the generated manifest (keep this output in the executed notebook)
!cat submission_manifest.json

{
  "github_username": "brainmegamind303-droid",
  "gpu_uuid": "GPU-fd8bb1a0-f2bf-2b19-956a-0e2661ff8a08",
  "gpu_name": "Tesla T4",
  "compute_capability": "7.5",
  "instance_seed": 2404270034748271502,
  "mpi_ranks": 4,
  "cartesian_dims": [
    2,
    2
  ],
  "global_grid": [
    512,
    512
  ],
  "local_padded_shape": [
    258,
    258
  ],
  "steps": 400,
  "dx": 0.001949317738791423,
  "dt": 3.1665330389724216e-06,
  "kappa_max": 0.23999727931416737,
  "dt_max": 3.958211169753244e-06,
  "lambda_damp": 0.15,
  "serialized_ms": 1050.654416000043,
  "overlapped_ms": 2164.406989000099,
  "serialized_cuda_event_ms": 1047.83447265625,
  "overlapped_cuda_event_ms": 2164.0152893066406,
  "parity_max_diff": 0.0,
  "parity_passed": true,
  "ptx_counts": {
    "st.shared.f64": 5,
    "ld.shared.f64": 5,
    "bar.sync": 1,
    "st.shared (any)": 5,
    "ld.shared (any)": 5
  },
  "ptx_audit_passed": true,
  "energy_trace_serialized": [
    {
      "step": 0,
      "energy": 0.08402847494

### Analytical & Forensic Engineering Responses

*(All numbers below come from my run: GPU = Tesla T4, UUID = GPU-fd8bb1a0-f2bf-2b19-956a-0e2661ff8a08, INSTANCE_SEED = 2404270034748271502.)*

- **Question 1 (Memory Coalescing & Strided Column Packing):**
  The local array is C-contiguous with shape `(258, 258)` `float64`, so element `(i, j)` lives at byte offset `8 * (258 * i + j)`. A halo **row** (`u[1, 1:257]` or `u[256, 1:257]`) is 256 consecutive `float64` values = 2,048 bytes: one contiguous DMA transfer, and a warp reading it issues coalesced memory transactions. A halo **column** (`u[1:257, 1]` and `u[1:257, 256]`) has consecutive elements separated by a stride of `258 * 8 = 2,064` bytes. In `pack_halos_kernel`, threads reading a column strided by 2,064 bytes cannot coalesce into a single contiguous DMA transfer. At warp level, the 32 lanes of a warp read 32 doubles that are 2,064 bytes apart, so each lane hits a different 32-byte sector (8 useful bytes per sector, ~25% efficiency) instead of 32 lanes covering 256 contiguous bytes as for a row; a single cudaMemcpy can only move one contiguous byte range, so a column would need either 256 separate copies or a pitched 2D copy.
  If an unstrided bounding-box copy were attempted over rows `1..256` from column `1` to `256` (`256 * 256` `float64` elements = `524,288` bytes for a single column vs `2,048` bytes of actual column data), that would transfer `522,240` extra bytes per column (or `1,044,480` extra bytes for left+right columns per rank per step). Across 400 steps and 4 ranks, that would waste about 1.67 GB of PCIe transfer volume (1,044,480 B x 400 steps x 4 ranks = 1,671,168,000 B), whereas packing into contiguous 1D `float64` buffers moves only `2 * 256 + 2 * 256 = 1,024` elements (`8,192` bytes) per rank per step.

- **Question 2 (Shared-Memory Tile Indexing & Bank Conflict Analysis):**
  For a `16 x 16` thread block loading an `18 x 18` `float64` `smem` tile:
  - Thread `(tx=0, ty=0)` writes its interior cell at `smem[1, 1]` and loads halo apron cells `smem[0, 1]` (up) and `smem[1, 0]` (left), then reads `smem[1, 1]`, `smem[0, 1]`, `smem[2, 1]`, `smem[1, 0]`, and `smem[1, 2]`.
  - Thread `(tx=15, ty=15)` writes its interior cell at `smem[16, 16]` and loads halo apron cells `smem[17, 16]` (down) and `smem[16, 17]` (right), then reads `smem[16, 16]`, `smem[15, 16]`, `smem[17, 16]`, `smem[16, 15]`, and `smem[16, 17]`.
  - On NVIDIA Compute Capability 7.5 (Tesla T4, 32 banks of 4 bytes each), each `float64` value occupies 8 bytes (2 consecutive banks). Accessing along a row across `tx` maps consecutive `float64` elements across distinct bank pairs with no bank conflicts within a half-warp, whereas striding down a column across `ty` with `SHARED_DIM = 18` (`18 * 8 = 144` bytes = 36 four-byte banks, `36 mod 32 = 4`) causes a **2-way bank conflict** across a 16-thread half-warp (thread k starts at bank `4k mod 32` and uses banks `4k, 4k+1`, so k = 8 wraps back to bank 0). In my actual kernel a warp is linearized as `ty*16 + tx`, so each 16-thread half-warp (the unit in which 64-bit shared accesses are serviced on CC 7.5) is exactly one tile row; real accesses (`smem[ty+1, tx+1]`, `smem[ty, tx+1]`, `smem[ty+2, tx+1]`, `smem[ty+1, tx]`, `smem[ty+1, tx+2]`) are therefore conflict-free, and the 2-way conflict applies only to a hypothetical column-wise walk.

- **Question 3 (PTX Assembly Audit & Warp Synchronization):**
  From `tiled_stencil_kernel.inspect_asm()` in my run:
  - `st.shared.f64`: **5**
  - `ld.shared.f64`: **5**
  - `bar.sync`: **1**
  - `ptx_audit_passed`: **True**
  A `16 x 16` thread block has 256 threads = 8 warps. Because warps execute asynchronously and neighbor cells in `smem` are written by different threads/warps (including halo apron threads), omitting `cuda.syncthreads()` (`bar.sync`) between writing `smem` and reading `smem` for the stencil causes a Read-After-Write (RAW) data hazard and corrupts results.

- **Question 4 (Multi-Process Context Sharing on a Single Physical GPU):**
  Launching `mpirun -np 4` creates 4 separate OS processes, each with its own CUDA context sharing the single Tesla T4 GPU's SMs, copy engines, and PCIe bandwidth via context scheduling/time-slicing.
  In my benchmark:
  - `serialized_ms` (wall, slowest rank): **1050.654 ms** (`serialized_cuda_event_ms`: **1047.834 ms**)
  - `overlapped_ms` (wall, slowest rank): **2164.407 ms** (`overlapped_cuda_event_ms`: **2164.015 ms**)
  - `overlap speedup (wall)`: **0.4854x**
  On a single shared T4 GPU, 4 processes contend for the same GPU compute/DMA resources, and launching 4 separate boundary-ring stencil kernels plus stream synchronizations adds launch and synchronization overhead on small boundary grids (`Grid size 8` / `16`). By contrast, on four dedicated GPUs connected via InfiniBand, each rank has dedicated GPU compute and fabric communication hardware, making genuine compute-communication overlap much more advantageous. The measured speedup of 0.4854x means the overlapped mode was about 2.06x slower than the serialized one in my run: kernels from different CUDA contexts are time-sliced rather than run concurrently, so `stream_interior` cannot truly run in parallel with other ranks' kernels; the overlapped step also issues 5 small stencil launches instead of 1 plus two stream synchronizations per step; and the halo payload is only 8,192 bytes per rank over intra-node MPI, so there is very little communication latency to hide. These are the most likely causes, not separately profiled.

- **Question 5 (Point-to-Point Message Deadlock & `MPI.PROC_NULL` Semantics):**
  If all ranks post blocking `Send` before blocking `Recv` in a fixed order and message sizes exceed MPI's eager-protocol buffer threshold, MPI switches to rendezvous mode where `Send` blocks until a matching `Recv` is posted, resulting in circular deadlock across ranks: neighbouring ranks each sit in `Send` waiting for the other's `Recv`, which is never reached (a cycle in the wait-for graph). In this lab the halo messages are only 256 * 8 = 2,048 bytes, which is typically below the eager limit, so the code could appear to work at this size and fail only on larger grids. Posting `Irecv` first, then `Isend`, then `MPI.Request.Waitall` (as in `exchange_halos_async`) removes the cycle.
  Using `MPI.PROC_NULL` treats outer non-existent neighbors as immediate no-op endpoints (`up=-2, down=2, left=-2, right=1` for rank 0 at `(0,0)`), simplifying neighbor exchange calls while keeping homogeneous Dirichlet boundary conditions (`u = 0.0`) on outer domain edges.

- **Question 6 (Explicit CFL Stability & Energy Conservation Proof):**
  With $\Delta x = \Delta y = 1/513 \approx 1.949318 \times 10^{-3}$, $\kappa_{\max} \approx 0.239997279$:
  - $\Delta t_{\max} = \frac{\Delta x^2}{4 \cdot \kappa_{\max}} \approx 3.958211 \times 10^{-6}\text{ s}$
  - $\Delta t_{\text{used}} \approx 3.166533 \times 10^{-6}\text{ s}$ ($\Delta t / \Delta t_{\max} = 0.8000 < 1$)
  Allreduce global energy trace (`E = sum(u * dx * dy)`):
  - Step 0: `8.4028474940e-02`
  - Step 100: `8.4018321079e-02`
  - Step 200: `8.4010680184e-02`
  - Step 300: `8.4003344598e-02`
  - Step 400: `8.3996044648e-02`
  Energy decreases monotonically (`energy_monotonic_decreasing: True`) and `parity_max_diff = 0.0` (`< 1e-12`), confirming numerical stability and exact parity between serialized and overlapped runs.